In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mvpoolfiles/sparse_softmax.py
/kaggle/input/mvpoolfiles/layers.py
/kaggle/input/mvpoolfiles/main.py
/kaggle/input/mvpoolfiles/models.py
/kaggle/input/proteins/PROTEINS/PROTEINS_graph_labels.txt
/kaggle/input/proteins/PROTEINS/PROTEINS_A.txt
/kaggle/input/proteins/PROTEINS/PROTEINS_node_attributes.txt
/kaggle/input/proteins/PROTEINS/PROTEINS_node_labels.txt
/kaggle/input/proteins/PROTEINS/PROTEINS_graph_indicator.txt


In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [3]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.0.0+cu121.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu121.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.0.0+cu121.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.0+cu121.html

Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu121.html
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu121.html
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu121.html
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu121.html


In [4]:
!pip install torch-geometric

In [5]:
!ls -R /kaggle/input/proteins


/kaggle/input/proteins:
PROTEINS

/kaggle/input/proteins/PROTEINS:
PROTEINS_A.txt		      PROTEINS_node_attributes.txt
PROTEINS_graph_indicator.txt  PROTEINS_node_labels.txt
PROTEINS_graph_labels.txt


In [6]:
import torch
import torch_geometric

print("PyTorch version:", torch.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("CUDA availability:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)


PyTorch version: 2.5.1+cu121
PyTorch Geometric version: 2.6.1
CUDA availability: True
CUDA version: 12.1


In [7]:
import shutil
import os

# Define paths
input_path = "/kaggle/input/proteins"  # This is a directory, not a file
working_dir = "/kaggle/working/traintest1"

# Ensure the working directory exists
os.makedirs(working_dir, exist_ok=True)

# Copy the entire directory to /kaggle/working/
shutil.copytree(input_path, working_dir, dirs_exist_ok=True)

print(f"Dataset successfully copied to {working_dir}")

Dataset successfully copied to /kaggle/working/traintest1


In [8]:
import os

dataset_path = "/kaggle/working/traintest1/"

# List all files and folders in dataset_path
if os.path.exists(dataset_path):
    print("Contents of dataset_path:", os.listdir(dataset_path))
else:
    print("Dataset path does not exist!")

Contents of dataset_path: ['test', 'train', 'PROTEINS']


In [9]:
import os

raw_dir_path = "/kaggle/working/traintest1/PROTEINS"
print("Contents of traintest files:", os.listdir(raw_dir_path))

Contents of traintest files: ['PROTEINS_graph_labels.txt', 'PROTEINS_A.txt', 'PROTEINS_node_attributes.txt', 'PROTEINS_graph_indicator.txt', 'test', 'train', 'PROTEINS', 'PROTEINS_node_labels.txt', 'processed']


In [10]:
file_path = "/kaggle/working/traintest1/PROTEINS/PROTEINS_A.txt"
with open(file_path, "r") as f:
    for _ in range(5):  # Print first 5 lines
        print(f.readline().strip())

12, 1
23, 1
33, 1
24, 2
32, 2


In [11]:
"""# Re-import necessary modules due to kernel reset
import os
import numpy as np
from sklearn.model_selection import train_test_split

# Re-define paths
proteins_inner_dir = "/kaggle/working/traintest1/PROTEINS"
A_path = os.path.join(proteins_inner_dir, "PROTEINS_A.txt")
indicator_path = os.path.join(proteins_inner_dir, "PROTEINS_graph_indicator.txt")
labels_path = os.path.join(proteins_inner_dir, "PROTEINS_graph_labels.txt")
node_labels_path = os.path.join(proteins_inner_dir, "PROTEINS_node_labels.txt")
node_attrs_path = os.path.join(proteins_inner_dir, "PROTEINS_node_attributes.txt")

# Load data
edge_list = np.loadtxt(A_path, dtype=int, delimiter=",")
graph_indicator = np.loadtxt(indicator_path, dtype=int)
graph_labels = np.loadtxt(labels_path, dtype=int)
node_labels = np.loadtxt(node_labels_path, dtype=int)
node_attributes = np.loadtxt(node_attrs_path, dtype=float)

# Normalize graph labels to start from 0
graph_labels -= graph_labels.min()

# Number of graphs
num_graphs = len(np.unique(graph_indicator))
graph_ids = np.arange(1, num_graphs + 1)

# Split graphs stratified by label
train_graphs, test_graphs = train_test_split(
    graph_ids,
    test_size=0.2,
    random_state=42,
    stratify=graph_labels
)

# Directories for output
base_dir = "/kaggle/working/traintest1/PROTEINS"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Save function with node labels and attributes
def save_split(graph_ids, edge_list, graph_indicator, graph_labels,
               node_labels, node_attributes, out_dir):
    graph_ids_zero_based = graph_ids - 1
    node_mask = np.isin(graph_indicator, graph_ids)
    node_indices = np.where(node_mask)[0] + 1  # Original node IDs (1-based)

    new_graph_indicator = graph_indicator[node_mask]
    new_graph_labels = graph_labels[graph_ids_zero_based]
    new_node_labels = node_labels[node_mask]
    new_node_attrs = node_attributes[node_mask]

    # Map old node indices to new sequential ones
    index_map = {old: new for new, old in enumerate(node_indices)}
    filtered_edges = edge_list[
        np.isin(edge_list[:, 0], node_indices) & np.isin(edge_list[:, 1], node_indices)
    ]
    remapped_edges = np.vectorize(index_map.get)(filtered_edges)

    # Save files
    np.savetxt(os.path.join(out_dir, "DS_A.txt"), remapped_edges, fmt="%d")
    np.savetxt(os.path.join(out_dir, "DS_graph_indicator.txt"), new_graph_indicator, fmt="%d")
    np.savetxt(os.path.join(out_dir, "DS_graph_labels.txt"), new_graph_labels, fmt="%d")
    np.savetxt(os.path.join(out_dir, "DS_node_labels.txt"), new_node_labels, fmt="%d")
    np.savetxt(os.path.join(out_dir, "DS_node_attributes.txt"), new_node_attrs, fmt="%.6f")

    print(f"Saved {len(graph_ids)} graphs and {len(new_graph_indicator)} nodes to {out_dir}")

# Save the train/test splits
save_split(train_graphs, edge_list, graph_indicator, graph_labels,
           node_labels, node_attributes, train_dir)
save_split(test_graphs, edge_list, graph_indicator, graph_labels,
           node_labels, node_attributes, test_dir)

base_dir
"""

'# Re-import necessary modules due to kernel reset\nimport os\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\n\n# Re-define paths\nproteins_inner_dir = "/kaggle/working/traintest1/PROTEINS"\nA_path = os.path.join(proteins_inner_dir, "PROTEINS_A.txt")\nindicator_path = os.path.join(proteins_inner_dir, "PROTEINS_graph_indicator.txt")\nlabels_path = os.path.join(proteins_inner_dir, "PROTEINS_graph_labels.txt")\nnode_labels_path = os.path.join(proteins_inner_dir, "PROTEINS_node_labels.txt")\nnode_attrs_path = os.path.join(proteins_inner_dir, "PROTEINS_node_attributes.txt")\n\n# Load data\nedge_list = np.loadtxt(A_path, dtype=int, delimiter=",")\ngraph_indicator = np.loadtxt(indicator_path, dtype=int)\ngraph_labels = np.loadtxt(labels_path, dtype=int)\nnode_labels = np.loadtxt(node_labels_path, dtype=int)\nnode_attributes = np.loadtxt(node_attrs_path, dtype=float)\n\n# Normalize graph labels to start from 0\ngraph_labels -= graph_labels.min()\n\n# Number of graphs

In [12]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [13]:
import argparse
import os
import sys
import torch
import torch.nn.functional as F
from torch_geometric.data import InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import TUDataset
from torch.utils.data import random_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report


# === Use correct sys.path for custom modules ===
sys.path.append('/kaggle/input/mvpoolfiles')  # Replace with your actual model location

# === Import your model ===
from models import Model

# === Define Custom Dataset Class ===
class CustomSplitTUDataset(InMemoryDataset):
    def __init__(self, root, name='PROTEINS', use_node_attr=True, transform=None, pre_transform=None):
        self.name = name
        self.use_node_attr = use_node_attr
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass

    def process(self):
        dataset = TUDataset(root=self.root.replace(f'/{self.name}', ''), name=self.name, use_node_attr=self.use_node_attr)
        torch.save(self.collate(dataset), self.processed_paths[0])

# === Handle Jupyter/Kaggle args ===
# If running in Kaggle or Jupyter environment, we need to manually set args
if 'ipykernel_launcher' in sys.argv[0]:
    sys.argv = ['main.py', '--dataset', 'PROTEINS', '--epochs', '100']  # Default args for Kaggle/Colab

# Remove the IPython/Jupyter argument
sys.argv = sys.argv[:1]

# Remove problematic Jupyter arguments
filtered_args = [arg for arg in sys.argv if not arg.startswith("-f")]
# Create an argument parser
parser = argparse.ArgumentParser()
parser.add_argument('--seed', type=int, default=777, help='random seed')
parser.add_argument('--batch_size', type=int, default=512, help='batch size')
parser.add_argument('--lr', type=float, default=0.001, help='learning rate')
parser.add_argument('--weight_decay', type=float, default=0.001, help='weight decay')
parser.add_argument('--nhid', type=int, default=128, help='hidden size')
parser.add_argument('--sample_neighbor', type=bool, default=True, help='whether sample neighbors')
parser.add_argument('--sparse_attention', type=bool, default=True, help='whether use sparse attention')
parser.add_argument('--structure_learning', type=bool, default=True, help='whether perform structure learning')
parser.add_argument('--pooling_ratio', type=float, default=0.5, help='pooling ratio')
parser.add_argument('--dropout_ratio', type=float, default=0.0, help='dropout ratio')
parser.add_argument('--lamb', type=float, default=1.0, help='trade-off parameter')
parser.add_argument('--dataset', type=str, default='PROTEINS', help='Dataset name')
parser.add_argument('--device', type=str, default='cuda', help='Device (cpu/cuda)')
parser.add_argument('--epochs', type=int, default=1000, help='Maximum epochs')
parser.add_argument('--patience', type=int, default=100, help='Early stopping patience')

# Use parse_args() since we have already removed unwanted args
args = parser.parse_args()
# Parse arguments, ignoring unknown ones

args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(args.seed)
print("Arguments successfully parsed.")
print(args)


# === Dataset Paths ===
train_path = '/kaggle/working/traintest1/PROTEINS/train'
test_path = '/kaggle/working/traintest1/PROTEINS/test'

train_dataset = CustomSplitTUDataset(root=train_path, name='PROTEINS')
test_dataset = CustomSplitTUDataset(root=test_path, name='PROTEINS')

args.num_classes = train_dataset.num_classes
args.num_features = train_dataset.num_features

# === Train/Val Split ===
num_train = int(len(train_dataset) * 0.9)
num_val = len(train_dataset) - num_train
train_data, val_data = random_split(train_dataset, [num_train, num_val])

train_loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=args.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

# === Model & Optimizer ===
print("Using device:", args.device)
model = Model(args).to(args.device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

# === Training Function ===
def train():
    min_loss = float("inf")
    patience_cnt = 0
    val_loss_values = []
    best_epoch = 0

    for epoch in range(args.epochs):
        model.train()
        loss_train = 0.0
        correct = 0

        for data in train_loader:
            optimizer.zero_grad()
            data = data.to(args.device)

            out = model(data)
            if out.shape[0] != data.y.shape[0]:
                print(f"Skipping batch: out.shape={out.shape}, data.y.shape={data.y.shape}")
                continue

            out = F.log_softmax(out, dim=1)
            loss = F.nll_loss(out, data.y)
            loss.backward()
            optimizer.step()

            loss_train += loss.item()
            pred = out.argmax(dim=1)
            correct += pred.eq(data.y).sum().item()

        acc_train = correct / len(train_loader.dataset)
        acc_val, loss_val = evaluate(val_loader)

        print(f"Epoch {epoch + 1}: TrainLoss={loss_train:.4f}, TrainAcc={acc_train:.4f}, ValLoss={loss_val:.4f}, ValAcc={acc_val:.4f}")
        val_loss_values.append(loss_val)

        torch.save(model.state_dict(), f'checkpoint_{epoch}.pth')

        if loss_val < min_loss:
            min_loss = loss_val
            best_epoch = epoch
            patience_cnt = 0
        else:
            patience_cnt += 1

        if patience_cnt >= args.patience:
            break

    print("Training done. Best epoch:", best_epoch)
    return best_epoch

# === Evaluation Function ===
'''def evaluate(loader):
    model.eval()
    correct = 0.0
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for data in loader:
            data = data.to(args.device)
            out = model(data)

            if out.shape[0] != data.y.shape[0]:
                continue

            out = F.log_softmax(out, dim=1)
            total_loss += F.nll_loss(out, data.y, reduction="sum").item()

            pred = out.argmax(dim=1)
            correct += pred.eq(data.y).sum().item()
            total_samples += data.y.size(0)

    acc = correct / total_samples if total_samples > 0 else 0
    avg_loss = total_loss / total_samples if total_samples > 0 else 0
    return acc, avg_loss'''
def evaluate(loader):
    model.eval()
    correct = 0.0
    total_loss = 0.0
    total_samples = 0
    true_labels = []
    pred_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(args.device)
            out = model(data)

            if out.shape[0] != data.y.shape[0]:
                continue

            out = F.log_softmax(out, dim=1)
            total_loss += F.nll_loss(out, data.y, reduction="sum").item()

            pred = out.argmax(dim=1)
            correct += pred.eq(data.y).sum().item()
            total_samples += data.y.size(0)

            true_labels.extend(data.y.cpu().numpy())  # Collect true labels
            pred_labels.extend(pred.cpu().numpy())   # Collect predicted labels

    acc = correct / total_samples if total_samples > 0 else 0
    avg_loss = total_loss / total_samples if total_samples > 0 else 0

    # Calculate additional metrics
    precision = precision_score(true_labels, pred_labels, average='macro')  # or 'weighted' if classes are imbalanced
    recall = recall_score(true_labels, pred_labels, average='macro')
    f1 = f1_score(true_labels, pred_labels, average='macro')
    conf_matrix = confusion_matrix(true_labels, pred_labels)
    class_report = classification_report(true_labels, pred_labels)

    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-Score: {f1:.4f}')
    print(f'Confusion Matrix:\n{conf_matrix}')
    print(f'Classification Report:\n{class_report}')

    return acc, avg_loss

# === Main Runner ===
if __name__ == "__main__":
    best_epoch = train()

    # Load best model & evaluate on test set
    model.load_state_dict(torch.load(f'checkpoint_{best_epoch}.pth'))
    test_acc, test_loss = evaluate(test_loader)
    print(f"Test Results: loss={test_loss:.4f}, accuracy={test_acc:.4f}")


Arguments successfully parsed.
Namespace(seed=777, batch_size=512, lr=0.001, weight_decay=0.001, nhid=128, sample_neighbor=True, sparse_attention=True, structure_learning=True, pooling_ratio=0.5, dropout_ratio=0.0, lamb=1.0, dataset='PROTEINS', device=device(type='cuda'), epochs=1000, patience=100)
Using device: cuda


/tmp/ipykernel_1566/2209884610.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


x1 shape: torch.Size([19075, 256]), x2 shape: torch.Size([19075, 256]), x3 shape: torch.Size([19075, 256])
x1 shape: torch.Size([19791, 256]), x2 shape: torch.Size([19791, 256]), x3 shape: torch.Size([19791, 256])
x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 256]), x3 shape: torch.Size([4605, 256])
Precision: 0.3482
Recall: 0.5000
F1-Score: 0.4105
Confusion Matrix:
[[78  0]
 [34  0]]
Classification Report:
              precision    recall  f1-score   support

           0       0.70      1.00      0.82        78
           1       0.00      0.00      0.00        34

    accuracy                           0.70       112
   macro avg       0.35      0.50      0.41       112
weighted avg       0.49      0.70      0.57       112

Epoch 1: TrainLoss=2.9700, TrainAcc=0.5025, ValLoss=1.2017, ValAcc=0.6964
x1 shape: torch.Size([20339, 256]), x2 shape: torch.Size([20339, 256]), x3 shape: torch.Size([20339, 256])
x1 shape: torch.Size([18527, 256]), x2 shape: torch.Size([18527,

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:13

x1 shape: torch.Size([18738, 256]), x2 shape: torch.Size([18738, 256]), x3 shape: torch.Size([18738, 256])
x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 256]), x3 shape: torch.Size([4605, 256])
Precision: 0.1518
Recall: 0.5000
F1-Score: 0.2329
Confusion Matrix:
[[ 0 78]
 [ 0 34]]
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        78
           1       0.30      1.00      0.47        34

    accuracy                           0.30       112
   macro avg       0.15      0.50      0.23       112
weighted avg       0.09      0.30      0.14       112

Epoch 3: TrainLoss=1.6440, TrainAcc=0.5065, ValLoss=1.1291, ValAcc=0.3036
x1 shape: torch.Size([19729, 256]), x2 shape: torch.Size([19729, 256]), x3 shape: torch.Size([19729, 256])
x1 shape: torch.Size([19137, 256]), x2 shape: torch.Size([19137, 256]), x3 shape: torch.Size([19137, 256])
x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 2

...

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 256]), x3 shape: torch.Size([4605, 256])
Precision: 0.6560
Recall: 0.5192
F1-Score: 0.2748
Confusion Matrix:
[[ 3 75]
 [ 0 34]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.04      0.07        78
           1       0.31      1.00      0.48        34

    accuracy                           0.33       112
   macro avg       0.66      0.52      0.27       112
weighted avg       0.79      0.33      0.20       112

Epoch 8: TrainLoss=1.3502, TrainAcc=0.5874, ValLoss=0.7216, ValAcc=0.3304
x1 shape: torch.Size([19641, 256]), x2 shape: torch.Size([19641, 256]), x3 shape: torch.Size([19641, 256])
x1 shape: torch.Size([19225, 256]), x2 shape: torch.Size([19225, 256]), x3 shape: torch.Size([19225, 256])
x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 256]), x3 shape: torch.Size([4605, 256])
Precision: 0.5598
Recall: 0.5328
F1-Score: 0.3657
Confusion Matrix:

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:13

Precision: 0.3482
Recall: 0.5000
F1-Score: 0.4105
Confusion Matrix:
[[78  0]
 [34  0]]
Classification Report:
              precision    recall  f1-score   support

           0       0.70      1.00      0.82        78
           1       0.00      0.00      0.00        34

    accuracy                           0.70       112
   macro avg       0.35      0.50      0.41       112
weighted avg       0.49      0.70      0.57       112

Epoch 12: TrainLoss=1.3351, TrainAcc=0.5844, ValLoss=0.5844, ValAcc=0.6964
x1 shape: torch.Size([19647, 256]), x2 shape: torch.Size([19647, 256]), x3 shape: torch.Size([19647, 256])
x1 shape: torch.Size([19219, 256]), x2 shape: torch.Size([19219, 256]), x3 shape: torch.Size([19219, 256])
x1 shape: torch.Size([4605, 256]), x2 shape: torch.Size([4605, 256]), x3 shape: torch.Size([4605, 256])
Precision: 0.7315
Recall: 0.5377
F1-Score: 0.4929
Confusion Matrix:
[[77  1]
 [31  3]]
Classification Report:
              precision    recall  f1-score   support

     

/tmp/ipykernel_1566/2209884610.py:231: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'checkpoint_{best_epoch}.pth'))
